In [ ]:
import os
import torch
import matplotlib.pyplot as plt

import open3d as o3d
from pytorch3d.io import load_objs_as_meshes, load_obj
from pytorch3d.structures import Meshes
from pytorch3d.vis.plotly_vis import AxisArgs, plot_batch_individually, plot_scene
from pytorch3d.vis.texture_vis import texturesuv_image_matplotlib
from pytorch3d.renderer import (
    look_at_view_transform,
    FoVPerspectiveCameras,
    PointLights,
    DirectionalLights,
    Materials,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    SoftPhongShader,
    TexturesUV,
    TexturesVertex
)
import numpy as np
from pano_utils import cubemap_to_equirectangular, pano_to_pointcloud, DistShader, MultiHitHardPhongShader, NormalAngleShader, get_sampling_prob_from_normal_angle

In [ ]:
def merge_colored_meshes(mesh1, mesh2):
    verts1 = np.asarray(mesh1.vertices)
    verts2 = np.asarray(mesh2.vertices)
    tris1 = np.asarray(mesh1.triangles)
    tris2 = np.asarray(mesh2.triangles)
    cols1 = np.asarray(mesh1.vertex_colors)
    cols2 = np.asarray(mesh2.vertex_colors)

    # Shift mesh2 triangle indices
    tris2_shifted = tris2 + verts1.shape[0]

    # Merge
    merged = o3d.geometry.TriangleMesh()
    merged.vertices = o3d.utility.Vector3dVector(np.vstack([verts1, verts2]))
    merged.triangles = o3d.utility.Vector3iVector(np.vstack([tris1, tris2_shifted]))
    merged.vertex_colors = o3d.utility.Vector3dVector(np.vstack([cols1, cols2]))

    merged.compute_vertex_normals()
    return merged



In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

# Set paths768
DATA_DIR = "./demo_data"
obj_filename = os.path.join(DATA_DIR, "cow_mesh/cow.obj")
# obj_filename = os.path.join(DATA_DIR, "can/f82039689f504922995936c68484aa61.obj")

In [ ]:
mesh = load_objs_as_meshes([obj_filename], device=device)
num_verts = mesh.verts_packed().shape[0]
mesh = mesh.offset_verts(-mesh.verts_packed().mean(dim=0, keepdim=True).repeat(num_verts, 1))
mesh = mesh.scale_verts(1.0 / mesh.verts_packed().max(dim=0)[0])

In [ ]:
verts = mesh.verts_packed()
r2 = verts.norm(dim=1, keepdim=True) ** 2
inverted_verts = verts / r2

faces = mesh.faces_packed()
flipped_faces = faces[:, [0, 2, 1]]
verts_uvs = mesh.textures.verts_uvs_padded()  # (1, V, 2)
faces_uvs = mesh.textures.faces_uvs_padded()  # (1, F, 3)
flipped_fuvs = faces_uvs[..., [0, 2, 1]]

new_textures = TexturesUV(
    maps=mesh.textures._maps_padded,
    faces_uvs=flipped_fuvs.to(device),  # (1, F, 3)
    verts_uvs=verts_uvs.to(device),  # (1, V, 2)
)

inverted_mesh = Meshes(verts=[inverted_verts], faces=[flipped_faces], textures=new_textures)

In [ ]:
from pytorch3d.renderer import look_at_rotation
from pytorch3d.renderer.blending import BlendParams
# Define view directions
camera_dirs = torch.tensor([
    [-1, 0, 0],   # -X
    [1, 0, 0],    # +X
    [0, -1, 0],   # -Y
    [0, 1, 0],    # +Y
    [0, 0, -1],   # -Z
    [0, 0, 1],    # +Z
], device=device, dtype=torch.float32)

# Custom up vectors to avoid singularities
up_vectors = torch.tensor([
    [0, 1, 0],    # for -X
    [0, 1, 0],    # for +X
    [0, 0, 1],   # for -Y
    [0, 0, 1],    # for +Y (must not be parallel to +Y dir)
    [0, 1, 0],    # for -Z
    [0, 1, 0],    # for +Z
], device=device, dtype=torch.float32)
# up_vectors = torch.tensor([
#     [0, -1, 0],    # for -X
#     [0, 1, 0],    # for +X
#     [0, 0, -1],   # for -Y
#     [0, 0, 1],    # for +Y (must not be parallel to +Y dir)
#     [0, -1, 0],    # for -Z
#     [0, 1, 0],    # for +Z
# ], device=device, dtype=torch.float32)

# Compute rotation matrices
R = look_at_rotation(camera_dirs, up=up_vectors)
T = torch.zeros_like(camera_dirs)  # Camera is at the origin

cameras = FoVPerspectiveCameras(device=device, R=R, T=T, fov=90, znear=0.1, zfar=200)

max_hits = 5
image_size = 512
# Set up renderer
raster_settings = RasterizationSettings(image_size=image_size, blur_radius=0.0, faces_per_pixel=max_hits)
lights = PointLights(device=device, location=[[0, 0, 0.]])
renderer = MeshRenderer(
    rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
    # shader=SoftPhongShader(device=device, cameras=cameras, lights=lights)
    shader=MultiHitHardPhongShader(device=device, cameras=cameras, lights=lights, max_hits=max_hits)
)
blend_params = BlendParams(background_color=(-1, -1, -1))

# Render images
all_images = renderer(inverted_mesh.extend(6), cameras=cameras, lights=lights, blend_params=blend_params)  # (6, H, W, 3)faces.shape

In [ ]:
first_hit_images = all_images[0]
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
ax[0, 0].imshow(first_hit_images[0, ..., :3].cpu().numpy())
ax[0, 0].set_title("-X")
ax[0, 1].imshow(first_hit_images[1, ..., :3].cpu().numpy())
ax[0, 1].set_title("+X")
ax[0, 2].imshow(first_hit_images[2, ..., :3].cpu().numpy())
ax[0, 2].set_title("-Y")
ax[1, 0].imshow(first_hit_images[3, ..., :3].cpu().numpy())
ax[1, 0].set_title("+Y")
ax[1, 1].imshow(first_hit_images[4, ..., :3].cpu().numpy())
ax[1, 1].set_title("-Z")
ax[1, 2].imshow(first_hit_images[5, ..., :3].cpu().numpy())
ax[1, 2].set_title("+Z")
plt.tight_layout()

In [ ]:
from copy import deepcopy
all_panos = []
pano_valid_regions = []
for images in deepcopy(all_images):
    cube_images = images[..., :3].permute(0, 3, 1, 2)  # (6, C, H, W)
    cube_images[0] = torch.flip(cube_images[0], dims=(1,))
    cube_images[2] = torch.flip(cube_images[2], dims=(1,))
    cube_images[4] = torch.flip(cube_images[4], dims=(1, 2))
    cube_images[5] = torch.flip(cube_images[5], dims=(2,))
    this_pano = cubemap_to_equirectangular(cube_images, H=image_size, device=device)
    all_panos.append(this_pano)
    this_pano_valid_region = this_pano.mean(axis=0) > 0
    pano_valid_regions.append(this_pano_valid_region)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(all_panos[0].cpu().numpy().transpose(1, 2, 0))

In [ ]:
dist_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
    shader=DistShader(cameras=cameras, max_hits=max_hits)
)
dist_maps = dist_renderer(inverted_mesh.extend(6), cameras=cameras)  # (6, H, W, 1)
all_pano_dist = []
for this_dist_maps in dist_maps:
    this_dist_maps = this_dist_maps.squeeze(-1).permute(0, 1, 2)  # (6, H, W)
    # this_dist_maps[0] = torch.flip(this_dist_maps[0], dims=(1,))  # Flip -X face horizontally
    this_pano_dist = cubemap_to_equirectangular(
        this_dist_maps[:, None],  # 6, H, W -> 6, 1, H, W
        H=image_size, 
        device=device
    )[0]
    all_pano_dist.append(this_pano_dist)

In [ ]:
normal_angle_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
    shader=NormalAngleShader(cameras=cameras, max_hits=max_hits)
)
normal_angle_maps = normal_angle_renderer(inverted_mesh.extend(6), cameras=cameras)  # (6, H, W, 1)
all_pano_normal_angle = []
for this_normal_angle_maps in normal_angle_maps:
    this_normal_angle_maps = this_normal_angle_maps.squeeze(-1).permute(0, 1, 2)  # (6, 3, H)
    # this_normal_angle_maps[0] = torch.flip(this_normal_angle_maps[0], dims=(1,))  # Flip -X face horizontally
    this_pano_normal_angle = cubemap_to_equirectangular(
        this_normal_angle_maps[:, None],  # 6, H, W -> 6, 1, H, W
        H=image_size,
        device=device
    )[0]
    all_pano_normal_angle.append(this_pano_normal_angle)

In [ ]:
def pano_to_mesh_3d(pano_color, pano_dist, valid_mask=None, max_edge_ratio=1.5):
    """
    Convert layered panorama + distance maps into a 3D mesh by triangulating both
    intra-layer (xy plane) and inter-layer (z direction) neighbors.

    Args:
        pano_color: (L, 3, H, 2H) RGB panoramas
        pano_dist:  (L, H, 2H)    Distance-to-origin maps
        valid_mask: (L, H, 2H)    Optional validity masks
        max_edge_ratio: float     Reject triangles with long edges (relative to median)

    Returns:
        verts:  (N, 3) FloatTensor
        faces:  (M, 3) LongTensor
        colors: (N, 3) FloatTensor
    """
    L, C, H, W = pano_color.shape
    device = pano_color.device

    pano_color = pano_color.permute(0, 2, 3, 1)  # (L, H, W, 3)
    pano_dist = pano_dist                       # (L, H, W)

    if valid_mask is None:
        valid_mask = torch.isfinite(pano_dist) & (pano_dist > 1e-5)

    # Step 1: Compute direction map (spherical coordinates)
    phi = torch.linspace(0, np.pi, H, device=device).view(H, 1)
    theta = torch.linspace(-np.pi, np.pi, W, device=device).view(1, W)
    x = torch.sin(phi) * torch.sin(theta)
    y = torch.cos(phi).repeat(1, W)
    z = torch.sin(phi) * torch.cos(theta)
    dirs = torch.stack([x, y, z], dim=-1)  # (H, W, 3)

    dirs = dirs.unsqueeze(0).expand(L, -1, -1, -1)  # (L, H, W, 3)
    pts = dirs * pano_dist.unsqueeze(-1)           # (L, H, W, 3)

    # Step 2: flatten valid pixels to verts/colors
    index_map = -torch.ones((L, H, W), dtype=torch.long, device=device)
    valid_idx = valid_mask.nonzero(as_tuple=False)
    index_map[valid_mask] = torch.arange(valid_idx.shape[0], device=device)

    verts = pts[valid_mask]                 # (N, 3)
    colors = pano_color[valid_mask]         # (N, 3)

    faces = []

    # Helper: construct triangles between 4 corners (a quad) using 2 triangles
    def tri_from_quad(i00, i01, i10, i11):
        m_a = (i00 >= 0) & (i01 >= 0) & (i10 >= 0)
        m_b = (i11 >= 0) & (i01 >= 0) & (i10 >= 0)
        tri_a = torch.stack([i00[m_a], i01[m_a], i10[m_a]], dim=-1)
        tri_b = torch.stack([i11[m_b], i01[m_b], i10[m_b]], dim=-1)
        return [tri_a, tri_b]

    # Step 3: in-layer XY triangles
    y = torch.arange(H - 1, device=device)
    x = torch.arange(W, device=device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')

    for l in range(L):
        i00 = index_map[l, yy, xx]
        i01 = index_map[l, yy, (xx + 1) % W]
        i10 = index_map[l, yy + 1, xx]
        i11 = index_map[l, yy + 1, (xx + 1) % W]
        faces += tri_from_quad(i00, i01, i10, i11)

    # Step 4: inter-layer Z triangles (connect adjacent depth layers)
    z = torch.arange(L - 1, device=device)
    y = torch.arange(H - 1, device=device)
    x = torch.arange(W, device=device)
    zz, yy, xx = torch.meshgrid(z, y, x, indexing='ij')

    i0 = index_map[zz, yy, xx]
    i1 = index_map[zz + 1, yy, xx]
    for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1)]: #, (1, 1), (-1, -1), (1, -1), (-1, 1)]:
        i2 = index_map[zz + 1, yy + dy, (xx + dx) % W]

        mask1 = (i0 >= 0) & (i1 >= 0) & (i2 >= 0)
        tri1 = torch.stack([i0[mask1], i1[mask1], i2[mask1]], dim=-1)
        faces += [tri1]

    # i0 = index_map[zz, yy, xx]
    # i1 = index_map[zz + 1, yy, xx]
    # for dx, dy in [(1, 0), (0, 1), (-1, 0), (0, -1), (1, 1), (-1, -1), (1, -1), (-1, 1)]:
    #     i2 = index_map[zz, yy + dy, (xx + dx) % W]

    #     mask1 = (i0 >= 0) & (i1 >= 0) & (i2 >= 0)
    #     tri1 = torch.stack([i0[mask1], i1[mask1], i2[mask1]], dim=-1)
    #     faces += [tri1]
    # i0 = index_map[zz, yy, xx]
    # i1 = index_map[zz + 1, yy, xx]
    # i2 = index_map[zz + 1, yy, (xx + 1) % W]
    # i3 = index_map[zz + 1, (yy + 1) , xx]

    # mask1 = (i0 >= 0) & (i1 >= 0) & (i2 >= 0)
    # mask2 = (i0 >= 0) & (i1 >= 0) & (i3 >= 0)
    # mask3 = (i0 >= 0) & (i2 >= 0) & (i3 >= 0)
    # tri1 = torch.stack([i0[mask1], i1[mask1], i2[mask1]], dim=-1)
    # tri2 = torch.stack([i0[mask2], i1[mask2], i3[mask2]], dim=-1)
    # tri3 = torch.stack([i0[mask3], i2[mask3], i3[mask3]], dim=-1)
    # faces += [tri1, tri2, tri3]

    faces = torch.cat(faces, dim=0)  # (M, 3)

    # Step 5: optional filtering of long-edge triangles
    v0, v1, v2 = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    e1 = (v0 - v1).norm(dim=1)
    e2 = (v1 - v2).norm(dim=1)
    e3 = (v2 - v0).norm(dim=1)
    edge_lengths = torch.cat([e1, e2, e3])
    edge_median = edge_lengths.median()
    valid_tri = (e1 < edge_median * max_edge_ratio) & \
                (e2 < edge_median * max_edge_ratio) & \
                (e3 < edge_median * max_edge_ratio)

    faces = faces[valid_tri]

    return verts, faces, colors


In [ ]:
all_verts, all_faces, all_colors = pano_to_mesh_3d(
    torch.stack(all_panos, dim=0),  # (L, 3, H, 2H)
    torch.stack(all_pano_dist, dim=0),  # (L, H, 2H)
    valid_mask=torch.stack(pano_valid_regions, dim=0),  # (L, H, 2H)
    max_edge_ratio=3
)
all_verts = all_verts / (all_verts.norm(dim=1, keepdim=True) ** 2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(all_panos[0].cpu().numpy().transpose(1, 2, 0))

In [ ]:

mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
mesh.triangles = o3d.utility.Vector3iVector(all_faces.cpu().numpy())
mesh.vertex_colors = o3d.utility.Vector3dVector(all_colors.cpu().numpy())
mesh.compute_vertex_normals()

# mesh = mesh.filter_smooth_simple(number_of_iterations=5)
# mesh.remove_unreferenced_vertices()
# mesh.remove_degenerate_triangles()
# mesh.remove_duplicated_triangles()
mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=100_000)

In [ ]:
# Step 1: Create point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
pcd.colors = o3d.utility.Vector3dVector(all_colors.cpu().numpy())

pcd = pcd.voxel_down_sample(0.005)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=12, std_ratio=4.0)

# Step 2: Estimate normals
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=50)
)
pcd.orient_normals_consistent_tangent_plane(k=100)

# Step 3: Poisson reconstruction
mesh2, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(densities)
# mesh2.remove_vertices_by_mask(dens < np.quantile(dens, 0.01))

# Step 4: Optional: crop low-density noise
bbox = pcd.get_axis_aligned_bounding_box()
mesh2 = mesh2.crop(bbox)

In [ ]:
from copy import deepcopy
pcd_tree = o3d.geometry.KDTreeFlann(pcd)
mesh_verts = np.asarray(mesh2.vertices)
mask = []

for v in mesh_verts:
    [_, _, d2] = pcd_tree.search_knn_vector_3d(v, 1)
    mask.append(np.sqrt(d2[0]) < 0.2)

mask = np.array(mask)
mesh2_cp = deepcopy(mesh2)
mesh2_cp.remove_vertices_by_mask(~mask)

In [ ]:
import numpy as np
from pythreejs import *
from IPython.display import display

mesh_to_show = mesh2_cp

verts  = np.asarray(mesh_to_show.vertices,  dtype=np.float32)        # (V, 3)
faces  = np.asarray(mesh_to_show.triangles, dtype=np.uint32)         # (F, 3)
vcolors = np.asarray(mesh_to_show.vertex_colors, dtype=np.float32)

geometry = BufferGeometry(
    attributes={
        'position': BufferAttribute(verts,  normalized=False),
        'color'   : BufferAttribute(vcolors, normalized=False),
    },
    index=BufferAttribute(faces.flatten(), normalized=False)
)

material = MeshLambertMaterial(vertexColors='VertexColors',
                               side='DoubleSide')   # show both sides

mesh_obj = Mesh(geometry=geometry, material=material)

camera = PerspectiveCamera(position=[2, 2, 2], up=[0, 0, 1])
camera.lookAt([0, 0, 0])

lights = [
    AmbientLight(intensity=1.0),
    DirectionalLight(position=[4, 4, 6], intensity=0.7),
]

scene = Scene(children=[mesh_obj, camera, *lights])

controls  = OrbitControls(controlling=camera)
renderer  = Renderer(camera=camera, scene=scene,
                     controls=[controls], width=600, height=600)

display(renderer)

In [ ]:
all_sampling_prob = []
for this_pano_normal_angle in all_pano_normal_angle:
    this_sampling_prob = get_sampling_prob_from_normal_angle(this_pano_normal_angle.abs(), min_prob=0.5)
    all_sampling_prob.append(this_sampling_prob)

In [ ]:
all_verts = []
all_colors = []
for pano, pano_depth, valid_mask, sampling_prob in zip(all_panos, all_pano_dist, pano_valid_regions, all_sampling_prob):
    verts, colors = pano_to_pointcloud(pano, pano_depth, valid_mask)
    # turn insideout back
    verts_norm = verts.norm(dim=1, keepdim=True) ** 2
    verts = verts / verts_norm

    # subsampling according to sampling prob
    sampling_prob = sampling_prob[valid_mask]
    subsample_mask = torch.rand(sampling_prob.shape, device=device) < sampling_prob
    verts = verts[subsample_mask]
    colors = colors[subsample_mask]

    all_verts.extend(verts)
    all_colors.extend(colors)
all_verts = torch.stack(all_verts, dim=0)
all_colors = torch.stack(all_colors, dim=0)


In [ ]:
all_verts.shape, all_colors.shape

In [ ]:
import numpy as np
from pythreejs import *
from IPython.display import display

num_pts_to_show = 200000
rnd_idx = torch.randperm(all_verts.shape[0])
picked_idx = rnd_idx[:num_pts_to_show]
verts = all_verts[picked_idx]
colors = all_colors[picked_idx]

geometry = BufferGeometry(
    attributes={
        'position': BufferAttribute(verts.cpu().numpy(), normalized=False),
        'color': BufferAttribute(colors.cpu().numpy(), normalized=False),
    }
)

material = PointsMaterial(vertexColors='VertexColors', size=0.05)
points = Points(geometry=geometry, material=material)

camera = PerspectiveCamera(position=[2, 2, 2], up=[0, 0, 1])
camera.lookAt([0, 0, 0])

scene = Scene(children=[points, camera, AmbientLight(intensity=0.5)])
controller = OrbitControls(controlling=camera)

renderer = Renderer(camera=camera, scene=scene, controls=[controller], 
                    width=600, height=600)

display(renderer)


In [ ]:
import open3d as o3d

method = 'poisson'        #  'poisson' | 'bpa' | 'alpha'

# ------------------------------------------------------------
# 1 .  Load / pre‑process the point cloud (your original steps)
# ------------------------------------------------------------
pcd = o3d.geometry.PointCloud()
pcd.points  = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
pcd.colors  = o3d.utility.Vector3dVector(all_colors.cpu().numpy())

pcd = pcd.voxel_down_sample(0.005)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=12, std_ratio=4.0)

pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=50)
)
pcd.orient_normals_consistent_tangent_plane(k=100)

# ------------------------------------------------------------
# 2 .  Surface reconstruction  ⬇⬇⬇
# ------------------------------------------------------------
if method == 'poisson':
    # -------- Poisson (watertight, smooth) -------------------
    mesh, density = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=9
    )
    dens = np.asarray(density)
    # mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.012))

elif method == 'bpa':
    # -------- Ball Pivoting (keeps real holes) ---------------
    # choose radii ≈ 1–3 × mean NN distance
    avg_d = np.mean(pcd.compute_nearest_neighbor_distance())
    radii = [avg_d * f for f in (1.2, 1.8, 2.4)]
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd, o3d.utility.DoubleVector(radii)
    )                    # :contentReference[oaicite:0]{index=0}

elif method == 'alpha':
    # -------- α‑Shape (tunable tightness) --------------------
    alpha = 0.03         # ↓ tighter, ↑ looser; tune to your scale
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
        pcd, alpha
    )                    # :contentReference[oaicite:1]{index=1}

else:
    raise ValueError(f"Unknown method: {method}")

# ------------------------------------------------------------
# 3 .  Shared post‑processing
# ------------------------------------------------------------
mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())
mesh = mesh.filter_smooth_simple(number_of_iterations=5)
mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=50_000)

In [ ]:
import numpy as np
from pythreejs import *
from IPython.display import display


# ------------------------------------------------------------
# 1.  Assume you already have an Open3D mesh called `mesh`
#     If not, replace the two lines below with your own arrays
# ------------------------------------------------------------
verts  = np.asarray(mesh.vertices,  dtype=np.float32)        # (V, 3)
faces  = np.asarray(mesh.triangles, dtype=np.uint32)         # (F, 3)

if mesh.has_vertex_colors():
    vcolors = np.asarray(mesh.vertex_colors, dtype=np.float32)
else:                               # fall back to the originals
    vcolors = colors.cpu().numpy().astype(np.float32)


# ------------------------------------------------------------
# 2.  Create THREE.BufferGeometry with positions + indices (+ colours)
# ------------------------------------------------------------
geometry = BufferGeometry(
    attributes={
        'position': BufferAttribute(verts,  normalized=False),
        'color'   : BufferAttribute(vcolors, normalized=False),
    },
    index=BufferAttribute(faces.flatten(), normalized=False)
)

# ------------------------------------------------------------
# 3.  Visual appearance
#     MeshLambertMaterial gives diffuse shading that reacts to lights
# ------------------------------------------------------------
material = MeshLambertMaterial(vertexColors='VertexColors',
                               side='DoubleSide')   # show both sides

mesh_obj = Mesh(geometry=geometry, material=material)

# ------------------------------------------------------------
# 4.  Camera, lights, scene, renderer (mostly unchanged)
# ------------------------------------------------------------
camera = PerspectiveCamera(position=[2, 2, 2], up=[0, 0, 1])
camera.lookAt([0, 0, 0])

lights = [
    AmbientLight(intensity=1.0),
    DirectionalLight(position=[4, 4, 6], intensity=0.7),
]

scene = Scene(children=[mesh_obj, camera, *lights])

controls  = OrbitControls(controlling=camera)
renderer  = Renderer(camera=camera, scene=scene,
                     controls=[controls], width=600, height=600)

display(renderer)

In [ ]:
from pytorch3d.structures import Pointclouds
from pytorch3d.renderer import (
    PointsRasterizationSettings,
    PointsRasterizer,
    PointsRenderer,
    AlphaCompositor
)

num_pts_to_show = 100000
rnd_idx = torch.randperm(verts.shape[0])
picked_idx = rnd_idx[:num_pts_to_show]
verts = verts[picked_idx]
colors = colors[picked_idx]
pointcloud = Pointclouds(points=[verts], features=[colors])

R, T = look_at_view_transform(10, 30, 0)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T, znear=0.01)

# Define the settings for rasterization and shading. Here we set the output image to be of size
# 512x512. As we are rendering images for visualization purposes only we will set faces_per_pixel=1
# and blur_radius=0.0. Refer to raster_points.py for explanations of these parameters. 
raster_settings = PointsRasterizationSettings(
    image_size=512, 
    radius = 0.01,
    points_per_pixel = 10
)


# Create a points renderer by compositing points using an alpha compositor (nearer points
# are weighted more heavily). See [1] for an explanation.
rasterizer = PointsRasterizer(cameras=cameras, raster_settings=raster_settings)
renderer = PointsRenderer(
    rasterizer=rasterizer,
    compositor=AlphaCompositor()
)

In [ ]:
import open3d as o3d

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_verts.cpu().numpy())
pcd.colors = o3d.utility.Vector3dVector(all_colors.cpu().numpy())
pcd = pcd.voxel_down_sample(0.005)
pcd, _ = pcd.remove_statistical_outlier(
    nb_neighbors=12,
    std_ratio=5.0
)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=50)
)
pcd.orient_normals_consistent_tangent_plane(k=100)
mesh, density = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=9
)
# (a) Crop to original bounding box (optional but useful)
mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())

# (b) Laplacian or Taubin smoothing
mesh = mesh.filter_smooth_simple(number_of_iterations=5)

# (c) Remove tiny disconnected shells
mesh.remove_unreferenced_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()


# (d) Optional decimation to a lighter mesh
mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=50000)

In [ ]:
t_mesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
t_mesh = t_mesh.fill_holes()
mesh = t_mesh.to_legacy()

In [ ]:
images = renderer(pointcloud)
plt.figure(figsize=(10, 10))
plt.imshow(images[0, ..., :3].cpu().numpy())
plt.axis("off")

In [ ]:
all_verts = []
all_faces = []
all_colors = []

for pano, pano_dist, pano_valid_region in zip(all_panos, all_pano_dist, pano_valid_regions):
    verts, faces, colors = pano_to_mesh(
        pano,
        pano_dist,
        valid_mask=pano_valid_region,
        max_edge_ratio=5
    )
    all_faces.extend(faces + len(all_verts))
    all_verts.extend(verts)
    all_colors.extend(colors)
all_verts = torch.stack(all_verts, dim=0)
all_faces = torch.stack(all_faces, dim=0)
all_colors = torch.stack(all_colors, dim=0)
all_verts = all_verts / (all_verts.norm(dim=1, keepdim=True) ** 2)